# 0. Rename RHS Folders by Stim Waveform

Use this first utility block when you have a parent stimulation folder containing multiple RHS session folders. Paste the parent folder path, click **Preview Rename Plan**, and review the proposed names. Click **Apply Renames** only when the preview looks correct.

The suffix is decoded from the first recorded channel in each RHS folder that contains nonzero `stim_data`, so it can handle recordings with multiple amplifier channels and stimulation on only one channel. The folder name suffix is formatted like `A-026 stim cathodic first 100us 200uA 1Hz 30 pulses 999ms RP comp`.


### Helper code used by Block 0

The notebook is the user interface. The folder scanning, waveform decoding, and actual renaming live in [`rename_rhs_folders_by_stim_waveform.py`](rename_rhs_folders_by_stim_waveform.py).

- **Preview Rename Plan** reads each immediate RHS session folder and shows the old/new folder names without changing anything.
- **Apply Renames** applies the previewed plan only for folders marked `rename`.
- If you paste a single RHS session folder instead of its parent, the utility will rename that one folder.

- The suffix includes the detected stim channel, adds `999ms RP` from the inferred refractory gap, and adds `comp` when the RHS compliance-limit bit is present.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function0_rename_rhs_folders(globals())


# 1. Plot All Channel Data Wideband

Use this first block to generate the raw wideband preview inline in VS Code/Jupyter. Choose or paste the RHS data folder, choose channels and an optional time window, click **Generate Preview**, then click **Save PNG** only if you want to save the image into that same selected data folder.

To change the default folder that appears when this notebook starts, edit `DEFAULT_DATA_DIR` near the top of the code cell below.

### Helper code used by Block 1

The notebook is the user interface. The low-level RHS parsing and raw plot drawing live in [`plot_rhs_raw_wideband_with_stim_legend.py`](plot_rhs_raw_wideband_with_stim_legend.py).

- `resolve_channel_selection(...)` accepts `all` for every recorded amplifier channel in the selected RHS folder, or explicit entries like `A-014`, `A-014-16`, and `A-014, A-016`.
- `parse_time_window(...)` accepts `all` or entries like `0-10 s` to display only part of the recording.
- `read_rhs_folder(folder, channel)` reads and concatenates all `.rhs` files for each selected channel.
- Raw wideband amplifier samples are converted to microvolts with Intan scaling: `0.195 * (uint16 - 32768)`.
- RHS `stim_data` is decoded from the Intan stim bitfield into signed command current in `uA`.
- No LFP/band-pass filter is applied in Block 1. `MAX_POINTS` only controls the display envelope used to draw long raw traces efficiently.

In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function1_raw_wideband(globals())


# 2. Plot Bandpass Filtered Data

Use this second block to generate a frequency-selected plot from the same raw RHS data. Enter `all` to keep all recorded frequencies, or enter a bandpass range such as `200-400 Hz` or `400-6000 Hz`. You can also enter a signed amplitude window such as `-100 - 100 uV`, and an optional time window such as `10-20 s`.

Only samples inside the signed amplitude window and selected time window are shown, and the y-axis is zoomed to that same amplitude window. The preview is not saved until you click **Save PNG**.

### Helper code used by Block 2

This block uses [`plot_rhs_filtered_wideband.py`](plot_rhs_filtered_wideband.py) for the filter-specific work, while still using the channel parser and RHS reader from `plot_rhs_raw_wideband_with_stim_legend.py`.

- `parse_frequency_range(...)` parses `all` for all recorded frequencies, or entries like `200-400 Hz` for the bandpass filter.
- `parse_amplitude_range(...)` parses signed entries like `-100 - 100 uV` for the amplitude window and y-axis range.
- `resolve_channel_selection(...)` accepts `all` for every recorded amplifier channel in the selected RHS folder, or explicit entries like `A-014`, `A-014-16`, and `A-014, A-016`.
- `parse_time_window(...)` accepts `all` or entries like `10-20 s` for the displayed time window.
- `bandpass_filter_wideband(...)` keeps all recorded frequencies when Bandpass is `all`, or applies a zero-phase Butterworth bandpass for numeric ranges.
- `plot_filtered_wideband(...)` draws only the bandpass filtered samples inside the selected signed amplitude window and zooms the y-axis to that window.

In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function2_bandpass_filtered(globals())


# 3. Plot Stim-Triggered Bandpass Events

Use this third block to split the session into stim-triggered events based on RHS `stim_data` onsets, then place all events into one combined PNG. Each event panel is aligned to the exact RHS stim trigger at `0 s`, with a selectable pre-trigger window and post-trigger window. Events are arranged 3 per row so each row stays compact.

Function 3 uses the same response filters as Function 2: channels (`all` for every recorded channel, or explicit channels/ranges), Bandpass (`all` or a numeric range), signed amplitude window, pre time, post time, and max points. Stim onsets are detected from any recorded channel in the RHS folder that contains nonzero `stim_data`, even if that channel is not selected for display. `Train gap (ms)` controls how close stim pulses can be while still being grouped as one train/event.

Enter **Post time (ms)** as a single value like `500` to show `0-500 ms` after stim, or as a range like `20-300` to leave the first `20 ms` after stim blank in the response traces. A light grey vertical line marks the stim trigger at `0 s`. Use **Show stim current** to include or hide the red `stim_data` waveform row. The preview is not saved until you click **Save PNG**.


### Helper code used by Block 3

This block uses [`plot_rhs_stim_triggered_events.py`](plot_rhs_stim_triggered_events.py) for stim-triggered event detection and plotting, and [`wideband_function3_ui.py`](wideband_function3_ui.py) for the compact notebook UI.

- `build_stim_triggered_events(...)` groups nearby `stim_data` pulses into train/event onsets.
- `plot_stim_triggered_events_grid(...)` creates one combined grid plot with 3 stim events per row, aligns all panels to stim trigger time `0 s`, draws a light grey trigger line, and adds the matching red `stim_data` waveform below the selected response channels when **Show stim current** is enabled.
- `Pre time (ms)` controls how much data before the trigger is shown; `Post time (ms)` controls the displayed post-trigger response window.
- `default_stim_events_grid_output_path(...)` names the single combined event PNG.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function3_stim_triggered_events(globals())


Def:
train gap (ms) controls how close stim pulses can be while still being grouped as one stimulation train/event. For example, for a 100 Hz train, pulses are about 10 ms apart. With Train gap (ms) = 12, those 5 pulses are grouped into one event. In Function 3, pre/post time is relative to each stim trigger: Pre time = 100 ms and Post time = 500 ms shows -100 to +500 ms around each trigger; Post time = 20-300 ms leaves the first 20 ms after trigger blank in the response traces.


# 4. Plot Recorded Response Only

Use this block when you want to display only the recorded neural response across the full recording timeline, without requiring or showing stimulation current. Choose the RHS folder, response channels (`all` for every recorded channel, or explicit channels/ranges), Bandpass, signed Amplitude window, and absolute recording Time window, then generate and optionally save the PNG.


### Helper code used by Block 4

This block reuses [`plot_rhs_filtered_wideband.py`](plot_rhs_filtered_wideband.py) to apply the selected bandpass and amplitude window. It does not search for `stim_data`, does not mark a stimulation channel, and does not draw a stim-current row.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function4_recorded_response_only(globals())
